# Python and data for energy: load, check, resample and plot a year of hourly data

**Lab for lesson B2** of *Energy Systems, Step by Step*, by Honglin Li. About 60 minutes, plus the practice tasks at the end.

You will repeat the lesson's steps in pandas on real data: the Southwest Power Pool's (SPP) hourly demand, wind and solar for 2025, from the U.S. Energy Information Administration's Form EIA-930. Every step ends with an **auto-check cell**: an `assert` that stays silent when your result is right and stops with a message when it is not.

**How to run it**

- *Google Colab:* open a new notebook, choose *File → Upload notebook*, then upload `b02-python-data.json` and `town_year.json` with the folder icon on the left.
- *Locally:* Python 3 with pandas (and matplotlib for the plots); keep the two data files next to this notebook.

**The two data files** (download them from the lesson page)

| File | What it holds |
|---|---|
| `b02-python-data.json` | SPP's 2025 demand, wind and solar in MW: EIA's adjusted values before this course's cleaning (each fuel's columns summed, negatives set to 0) |
| `town_year.json` | the cleaned shapes Prairie Town uses: each series divided by its 2025 maximum, in thousandths |

EIA data are U.S. government works in the public domain; the derived files are CC BY 4.0. Prairie Town is fictional, and its farm sizes are illustrative.

In [ ]:
import json
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt   # pre-installed in Colab
    PLOTS = True
except ImportError:                   # pandas still runs every step; only the plots are skipped
    PLOTS = False
    print("matplotlib is not installed: the plots will be skipped.")

for name in ["b02-python-data.json", "town_year.json"]:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
print("pandas", pd.__version__, "· data files found")

## Part 1 · The raw file

### Step 1 · Load

A year of hourly data has 8,760 rows. The file stores the first time stamp and three lists of numbers. EIA-930 stamps each hour in **UTC**, at the **end** of the hour, so we build that index first and fix it in step 3.

In [ ]:
with open("b02-python-data.json") as f:
    raw = json.load(f)
print(raw["meta"]["source"])
print(raw["meta"]["time"])

ends = pd.date_range(raw["start_utc"], periods=len(raw["demand"]), freq="h", tz="UTC")
df = pd.DataFrame({c: raw[c] for c in ["demand", "wind", "solar"]}, index=ends)
print(df.shape)   # rows, columns
df.head()         # the first five rows

In [ ]:
# auto-check
assert df.shape == (8760, 3), "expected one row per hour of 2025 and three columns"
assert str(df.index.tz) == "UTC", "the index should still be in UTC here"
print("Check passed: one row per hour, three columns, UTC stamps.")

### Step 2 · Inspect

Summary statistics are the first check. One number cannot be right for a grid that serves millions of people. Find it.

In [ ]:
df.describe().round(0)   # count, mean, spread, quartiles, min and max of each column

In [ ]:
low = df["demand"].idxmin()   # when was demand lowest?
print(low, "→", df.loc[low, "demand"], "MW; the median is", df["demand"].median(), "MW")
df.loc[low - pd.Timedelta(hours=2): low + pd.Timedelta(hours=2), "demand"]

In [ ]:
# auto-check
assert df["demand"].min() == 1505, "the lowest demand should be the 1,505 MW spike"
assert df["demand"].min() < 0.05 * df["demand"].median(), "the spike is under 5% of the median"
print("Check passed: one hour of demand at 1,505 MW, almost certainly a reporting error (check it against the operator's records).")

### Step 3 · Fix the time stamps

Group the hours into days the wrong way first: by the UTC date of each stamp. The first and last days come out as fragments.

In [ ]:
df.groupby(df.index.date).size().value_counts()   # wrong: UTC days cut Central days in the evening

Now the right way: label each hour by its **start**, convert to U.S. Central time (with daylight saving), then group by local date.

In [ ]:
starts = df.index - pd.Timedelta(hours=1)          # label each hour by its start
df.index = starts.tz_convert("America/Chicago")    # UTC to Central time, with daylight saving
per_day = df.groupby(df.index.date).size()
print(per_day.value_counts())
per_day[per_day != 24]                             # the days that are not 24 hours long

In [ ]:
# auto-check
assert per_day.value_counts().to_dict() == {24: 363, 23: 1, 25: 1}
assert per_day[date(2025, 3, 9)] == 23 and per_day[date(2025, 11, 2)] == 25
print("Check passed: 9 March has 23 hours and 2 November 25. Both are real, not errors.")

### Step 4 · Find stuck sensors

A stuck sensor repeats its last reading. Flag every run of the same **non-zero** value that lasts at least `min_run` hours (zeros are fine: solar is zero every night). Try `min_run` = 2, 3 and 4 and find the value the data build used.

In [ ]:
def repeats(s, min_run=3):
    # True for hours inside a run of at least min_run identical non-zero values
    run = (s != s.shift()).cumsum()               # a new id each time the value changes
    length = s.groupby(run).transform("size")     # how long each run lasts
    return (length >= min_run) & (s != 0)         # long runs of one non-zero value

for n in [2, 3, 4]:
    print("min_run =", n, {c: int(repeats(df[c], n).sum()) for c in df})

In [ ]:
# auto-check: the data build flagged 671 wind hours and 1,377 solar hours
assert int(repeats(df["wind"]).sum()) == 671
assert int(repeats(df["solar"]).sum()) == 1377
print("Check passed: with min_run = 3 your counts match the data build.")

### Step 5 · Plot the worst run

Counts hide the story; a plot shows it. The longest run sits in December.

In [ ]:
def longest_repeat(d):
    # the longest run of one non-zero value, in hours, for each column of a DataFrame
    def one(s):
        length = s.groupby((s != s.shift()).cumsum()).transform("size")
        return int(length[s != 0].max())
    return d.apply(one)

print(longest_repeat(df))
dec = df.loc["2025-12-12":"2025-12-24"]
if PLOTS:
    dec.plot(subplots=True, figsize=(9, 6), title="SPP, 12 to 24 December 2025 (MW)")
    plt.show()
dec.loc["2025-12-16 00:00":"2025-12-16 05:00"]   # solar after midnight

In [ ]:
# auto-check
assert longest_repeat(df).to_dict() == {"demand": 155, "wind": 155, "solar": 155}
assert (dec.loc["2025-12-16 00:00":"2025-12-16 03:00", "solar"] == 617).all()
print("Check passed: every column froze for 155 hours; solar 'made' 617 MW after midnight.")

### Step 6 · A physics check

Repeats are one symptom; physics gives another. Between midnight and 4 a.m. local time, solar output must be near zero. Flag hours above 5% of the year's maximum.

In [ ]:
def bad_night(d):
    night = d.index.hour < 4                                # hours starting 00:00 to 03:00, local time
    return night & (d["solar"] > 0.05 * d["solar"].max())   # sun down, yet over 5% of the peak

bad_raw = bad_night(df)
print("raw file:", int(bad_raw.sum()), "night hours fail;",
      int((bad_raw & repeats(df["solar"])).sum()), "of them were already flagged as repeats")

In [ ]:
# auto-check
assert int(bad_raw.sum()) == 214
assert int((bad_raw & repeats(df["solar"])).sum()) == 198
print("Check passed: the night check catches 16 hours the repeat rule missed.")

## Part 2 · The cleaned file

`town_year.json` holds the shapes after the data build repaired the spike and the stuck sensors. Load it the same way, with the time fix built in, and divide by 1,000 so each series runs from 0 to 1 (a share of its 2025 maximum).

In [ ]:
with open("town_year.json") as f:
    ty = json.load(f)
ends = pd.date_range(ty["start_utc"], periods=len(ty["demand"]), freq="h", tz="UTC")
town = pd.DataFrame({c: ty[c] for c in ["demand", "wind", "solar"]},
                    index=(ends - pd.Timedelta(hours=1)).tz_convert("America/Chicago")) / 1000
town.describe().round(3)

In [ ]:
bad = bad_night(town)
print("cleaned file:", int(bad.sum()), "night hours fail")

In [ ]:
# auto-check
assert town.shape == (8760, 3) and (town.max() == 1).all()
assert int(bad.sum()) == 0
print("Check passed: the cleaned file has no solar at night.")

### How the data build fixes it

In late winter raw solar barely changed between day and night, so the repeat rule missed it and a same-hour fill could copy it. The data build adds a night rule, a documented heuristic for this SPP, Central-time example: in each month, an hour of the day whose **median** solar output is zero is night, and so are the hours from 22:00 to 05:00 all year; any solar output then is set to zero. (The build also zeroes hours when the sun is below the horizon at SPP's centre; this notebook leaves that part out.)

Apply the rule to the raw file and run the night check again. Hours set to zero pass that check by construction: it does not test the daytime values.

In [ ]:
raw_solar = df["solar"]
median = raw_solar.groupby([df.index.month, df.index.hour]).transform("median")   # each month's median for each hour of the day
night = (median == 0) | df.index.hour.isin([22, 23, 0, 1, 2, 3, 4])              # plus the hours when the sun is always down
df_fixed = df.assign(solar=raw_solar.where(~night, 0.0))
print("hours set to zero:", int((night & (raw_solar != 0)).sum()),
      "· night hours failing after the rule:", int(bad_night(df_fixed).sum()))

In [ ]:
# auto-check
assert int(bad_night(df_fixed).sum()) == 0
print("Check passed: with the physics rule, no solar at night.")

## Part 3 · Resample, plot and read a duration curve

Scale the shapes to Prairie Town (illustrative sizes, as in lesson 0): a 150 MW wind farm, a 200 MW solar farm and a 450 MW demand peak. Energy = average power × time, and each value covers one hour, so **energy is a sum** of the hourly MW values (in MWh) and **average power is a mean**.

In [ ]:
size = pd.Series({"demand": 450, "wind": 150, "solar": 200})   # MW, illustrative
town_mw = town * size
wind = town_mw["wind"]
daily = wind.resample("D").mean()                # MW: average power each day
monthly_gwh = wind.resample("MS").sum() / 1000   # GWh: energy each month
if PLOTS:
    fig, ax = plt.subplots(3, 1, figsize=(9, 8))
    wind.plot(ax=ax[0], lw=0.4, title="Wind farm, every hour (MW)")
    daily.plot(ax=ax[1], title="Daily mean (MW)")
    monthly_gwh.plot.bar(ax=ax[2], title="Energy each month (GWh)")
    plt.tight_layout()
    plt.show()
monthly_gwh.round(1)

In [ ]:
# auto-check
assert len(daily) == 365 and len(monthly_gwh) == 12
assert monthly_gwh.idxmax().month == 3 and round(monthly_gwh.max(), 1) == 70.7
assert monthly_gwh.idxmin().month == 9 and round(monthly_gwh.min(), 1) == 45.1
print("Check passed: most wind energy in March, least in September.")

In [ ]:
curve = town_mw["demand"].sort_values(ascending=False).reset_index(drop=True)
if PLOTS:
    curve.plot(title="Prairie Town demand, hours sorted from highest to lowest (MW)")
    plt.axhline(0.9 * size["demand"], ls="--", color="grey")
    plt.xlabel("hours in the year")
    plt.show()
hours_90 = int((town["demand"] > 0.9).sum())                       # hours above 90% of the peak
cf = {k: town_mw[k].sum() / (size[k] * len(town_mw)) for k in ["wind", "solar"]}   # energy ÷ (size × hours)
print("hours above 90% of the peak:", hours_90)
print("capacity factors:", {k: f"{100 * v:.1f}%" for k, v in cf.items()})

In [ ]:
# auto-check
assert hours_90 == 180
assert round(100 * cf["wind"], 1) == 52.8 and round(100 * cf["solar"], 1) == 23.4
print("Check passed. These capacity factors are illustrative: each farm reaches full output in SPP's best hour of 2025.")

## Part 4 · Asserts as a gate

Collect the checks in one place and run them on every file. A failing check is doing its job: it stops the work until someone decides what to do.

In [ ]:
CHECKS = {
    "8,760 rows":            lambda d: len(d) == 8760,
    "no missing values":     lambda d: bool(d.notna().all().all()),
    "no demand spike":       lambda d: bool(d["demand"].min() > 0.5 * d["demand"].median()),
    "no sensor stuck 6+ h":  lambda d: bool(longest_repeat(d).max() < 6),
    "no solar at night":     lambda d: bool(bad_night(d).sum() == 0),
}

def report(d):
    return pd.Series({name: rule(d) for name, rule in CHECKS.items()})

table = pd.DataFrame({"raw": report(df), "raw + night rule": report(df_fixed), "cleaned": report(town)})
table

In [ ]:
# auto-check, and the gate: analysis starts only from a file that passes every rule
assert (~table["raw"]).sum() == 3 and (~table["raw + night rule"]).sum() == 2 and (~table["cleaned"]).sum() == 0
for name, rule in CHECKS.items():
    assert rule(town), name
print("Check passed: the cleaned file passes every rule; the night rule alone does not fix the raw file's spike and repeats.")

## Your turn (practice, about 60 minutes)

These have no single right answer; write a sentence under each result saying what you conclude.

1. Change the night threshold from 5% to 1% of the peak. How many raw and cleaned hours fail now? Which threshold would you defend, and why?
2. Find the dates of the demand repeats (`repeats(df["demand"])`). Do they coincide with the wind and solar repeats? What does that suggest about their cause?
3. Compute SPP's monthly solar energy in the raw file before and after the night rule (`df` and `df_fixed`). Which months change, and by how much?
4. Write an assert that every month has between 672 and 744 hours. Run it, then explain why March has 743 hours and November 721.

In [ ]:
# 1. your code here

In [ ]:
# 2. your code here

In [ ]:
# 3. your code here

In [ ]:
# 4. your code here

---
**What you practised:** loading with an explicit time index, inspecting before analysing, fixing time zones and hour-ending stamps, three kinds of checks (range, repeats, physics), a documented fix, resampling with mean and sum, a duration curve, capacity factors, and asserts as a gate.

**Sources:** EIA, Hourly Electric Grid Monitor (Form EIA-930), six-month BALANCE files for 2025, balancing authority SWPP, accessed 2 November 2026; EIA, Form EIA-930 instructions (hourly integrated values by hour ending, UTC stamps).

Text CC BY 4.0, code MIT: teachers and students may reuse and adapt it, with credit to Honglin Li. Corrections: hlli@ksu.edu.